In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Parameters
num_records = 5000
start_time = datetime.now() - timedelta(days=7)

# Time series (every 5 mins)
timestamps = [start_time + timedelta(minutes=5*i) for i in range(num_records)]

data = {
    "timestamp": timestamps,
    "building_id": np.random.choice(["B1", "B2"], num_records),
    "floor": np.random.choice([1, 2, 3, 4, 5], num_records),
    "room": np.random.choice(
        ["101", "102", "201", "202", "ServerRoom", "MechanicalRoom"],
        num_records
    ),
    "zone": np.random.choice(["North", "South", "East", "West"], num_records),
    "equipment_id": np.random.choice(["HVAC_1", "HVAC_2", "PUMP_1"], num_records),
    "temperature_c": np.random.normal(22, 2, num_records),
    "pressure_bar": np.random.normal(2.5, 0.3, num_records),
    "vibration_mm_s": np.random.normal(1.2, 0.2, num_records),
    "power_kw": np.random.normal(18, 3, num_records),
    "humidity_pct": np.random.normal(45, 8, num_records)
}

df = pd.DataFrame(data)
df.head()

,timestamp,building_id,floor,room,zone,equipment_id,temperature_c,pressure_bar,vibration_mm_s,power_kw,humidity_pct
0,2026-04-07 09:50:41.924638,B1,2,MechanicalRoom,North,PUMP_1,21.142922,2.930193,1.116337,16.657604,47.641796
1,2026-04-07 09:55:41.924638,B2,5,102,West,HVAC_2,22.768872,2.020206,0.980910,21.682481,20.537834
2,2026-04-07 10:00:41.924638,B1,1,102,South,HVAC_2,19.940790,2.695958,1.437873,17.218308,50.070784
3,2026-04-07 10:05:41.924638,B1,5,101,South,PUMP_1,23.248729,2.203953,1.131370,24.102641,37.104208
4,2026-04-07 10:10:41.924638,B2,1,101,West,HVAC_2,25.297954,2.757680,1.007964,16.857018,63.018484


In [2]:
import pandas as pd

# Path to the uploaded CSV in the Lakehouse
file_path = "/lakehouse/default/Files/fake_data.csv"

# Load CSV into DataFrame
df = pd.read_csv(file_path)

# Preview data
df.head()

,timestamp,building_id,floor,room,zone,equipment_id,temperature_c,pressure_bar,vibration_mm_s,power_kw,humidity_pct
0,2026-04-07 13:04:48.587662,B2,5,202,East,PUMP_1,21.505200,2.239770,1.069976,12.910942,49.176822
1,2026-04-07 13:09:48.587662,B1,4,ServerRoom,South,PUMP_1,25.255063,2.435978,0.865027,20.866896,58.989394
2,2026-04-07 13:14:48.587662,B2,4,202,North,HVAC_2,21.650596,2.775705,1.372383,14.014077,39.730451
3,2026-04-07 13:19:48.587662,B1,5,101,South,HVAC_2,22.673286,2.248914,1.263111,15.983124,36.419066
4,2026-04-07 13:24:48.587662,B1,4,102,North,PUMP_1,22.869516,2.408140,1.400098,17.327511,45.338672


In [3]:
# Baseline and threshold definitions for all sensor features

THRESHOLDS = {
    "temperature_c": {
        "base": 20.0,     # Expected normal temperature
        "green": 0.5,     # ±0.5°C → Green
        "yellow": 5.0     # ±5°C   → Yellow, beyond → Red
    },
    "pressure_bar": {
        "base": 2.5,      # Normal system pressure
        "green": 0.3,
        "yellow": 1.0
    },
    "vibration_mm_s": {
        "base": 1.2,      # Normal vibration
        "green": 0.3,
        "yellow": 1.0
    },
    "power_kw": {
        "base": 18.0,     # Typical power consumption
        "green": 2.0,
        "yellow": 6.0
    },
    "humidity_pct": {
        "base": 45.0,     # Normal indoor humidity
        "green": 5.0,
        "yellow": 15.0
    }
}

THRESHOLDS

{'temperature_c': {'base': 20.0, 'green': 0.5, 'yellow': 5.0},
 'pressure_bar': {'base': 2.5, 'green': 0.3, 'yellow': 1.0},
 'vibration_mm_s': {'base': 1.2, 'green': 0.3, 'yellow': 1.0},
 'power_kw': {'base': 18.0, 'green': 2.0, 'yellow': 6.0},
 'humidity_pct': {'base': 45.0, 'green': 5.0, 'yellow': 15.0}}

In [4]:
# ✅ Traffic‑Light Function Code
def get_traffic_light_status(value, base, green_limit, yellow_limit):
    deviation = abs(value - base)
    
    if deviation <= green_limit:
        return "Green"
    elif deviation <= yellow_limit:
        return "Yellow"
    else:
        return "Red"

In [5]:
# Apply traffic-light status to temperature only

df["temp_status"] = df["temperature_c"].apply(
    lambda x: get_traffic_light_status(
        x,
        THRESHOLDS["temperature_c"]["base"],
        THRESHOLDS["temperature_c"]["green"],
        THRESHOLDS["temperature_c"]["yellow"]
    )
)

# Quick verification
print(df[["temperature_c", "temp_status"]].head())

   temperature_c temp_status
0      21.505200      Yellow
1      25.255063         Red
2      21.650596      Yellow
3      22.673286      Yellow
4      22.869516      Yellow


In [6]:
# Apply traffic-light status to pressure only

df["pressure_status"] = df["pressure_bar"].apply(
    lambda x: get_traffic_light_status(
        x,
        THRESHOLDS["pressure_bar"]["base"],
        THRESHOLDS["pressure_bar"]["green"],
        THRESHOLDS["pressure_bar"]["yellow"]
    )
)

# Quick verification
print(df[["pressure_bar", "pressure_status"]].head())

   pressure_bar pressure_status
0      2.239770           Green
1      2.435978           Green
2      2.775705           Green
3      2.248914           Green
4      2.408140           Green


In [7]:
# Apply traffic-light status to vibration only

df["vibration_status"] = df["vibration_mm_s"].apply(
    lambda x: get_traffic_light_status(
        x,
        THRESHOLDS["vibration_mm_s"]["base"],
        THRESHOLDS["vibration_mm_s"]["green"],
        THRESHOLDS["vibration_mm_s"]["yellow"]
    )
)

# Quick verification
print(df[["vibration_mm_s", "vibration_status"]].head())

   vibration_mm_s vibration_status
0        1.069976            Green
1        0.865027           Yellow
2        1.372383            Green
3        1.263111            Green
4        1.400098            Green


In [8]:
# Apply traffic-light status to power consumption only

df["power_status"] = df["power_kw"].apply(
    lambda x: get_traffic_light_status(
        x,
        THRESHOLDS["power_kw"]["base"],
        THRESHOLDS["power_kw"]["green"],
        THRESHOLDS["power_kw"]["yellow"]
    )
)

# Quick verification
print(df[["power_kw", "power_status"]].head())

    power_kw power_status
0  12.910942       Yellow
1  20.866896       Yellow
2  14.014077       Yellow
3  15.983124       Yellow
4  17.327511        Green


In [9]:
# Apply traffic-light status to humidity only

df["humidity_status"] = df["humidity_pct"].apply(
    lambda x: get_traffic_light_status(
        x,
        THRESHOLDS["humidity_pct"]["base"],
        THRESHOLDS["humidity_pct"]["green"],
        THRESHOLDS["humidity_pct"]["yellow"]
    )
)

# Quick verification
print(df[["humidity_pct", "humidity_status"]].head())

   humidity_pct humidity_status
0     49.176822           Green
1     58.989394          Yellow
2     39.730451          Yellow
3     36.419066          Yellow
4     45.338672           Green


In [10]:
# Create a unified anomaly flag based on all status columns

status_columns = [
    "temp_status",
    "pressure_status",
    "vibration_status",
    "power_status",
    "humidity_status"
]

df["is_anomaly"] = df[status_columns].isin(["Yellow", "Red"]).any(axis=1).astype(int)

# Quick verification
print(df[status_columns + ["is_anomaly"]].head())


  temp_status pressure_status vibration_status power_status humidity_status  \
0      Yellow           Green            Green       Yellow           Green   
1         Red           Green           Yellow       Yellow          Yellow   
2      Yellow           Green            Green       Yellow          Yellow   
3      Yellow           Green            Green       Yellow          Yellow   
4      Yellow           Green            Green        Green           Green   

   is_anomaly  
0           1  
1           1  
2           1  
3           1  
4           1  


In [11]:
# Export final monitoring output with traffic-light ratings to CSV

output_path = "/lakehouse/default/Files/facilities_monitoring_output.csv"

df.to_csv(output_path, index=False)

output_path


'/lakehouse/default/Files/facilities_monitoring_output.csv'